# Evidencia de Aprendizaje 3 — Optimización de procesos de desarrollo
### Automatización con GitHub Actions y despliegue de un servicio en la nube

**Jeremy Iván Pedraza Hernández** · Programación para Análisis de Datos · IU Digital de Antioquia

- Repositorio: https://github.com/Jeremy-Pedraza/Programacion-Analisis-Datos
- Aplicación web desplegada (Render): https://galaxy-a55-analytics.onrender.com

Este cuaderno es un artefacto **ejecutable**: en lugar de describir el proyecto, lo pone en marcha. Recorre el historial Git, ejecuta las pruebas, arranca la aplicación web y consulta sus *endpoints*, y muestra los *workflows* de CI/CD. Funciona tanto en local (dentro del repositorio) como en Google Colab (clona el repositorio automáticamente).

## Fase 0 — Configuración del entorno

Detecta si el cuaderno se ejecuta dentro del repositorio o en Colab; en Colab clona el repositorio. Luego instala el paquete y sus dependencias de desarrollo.

In [1]:
import os, subprocess, sys
from pathlib import Path

def encontrar_raiz():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'ml_scraper' / 'web.py').exists():
            return cand
    return None

RAIZ = encontrar_raiz()
if RAIZ is None:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Jeremy-Pedraza/Programacion-Analisis-Datos.git'], check=True)
    RAIZ = Path.cwd() / 'Programacion-Analisis-Datos'
os.chdir(RAIZ)
print('Raíz del proyecto:', RAIZ)

Raíz del proyecto: C:\Users\Desarrollador\AppData\Local\Temp\claude\f2e751d665dc1502\96d621c8-25e2-4004-af79-f64fdf40813c\scratchpad\ev3\repo


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements-dev.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Dependencias instaladas.')

Dependencias instaladas.


## Fase 1 — Control de versiones (Git)

El proyecto se construyó con un modelo de ramas (`main`, `develop`, `feature/*`, `docs/*`) e integraciones por *pull request*. Las celdas muestran el grafo de commits y la convención *Conventional Commits*.

In [3]:
import subprocess
def git(*args):
    return subprocess.run(['git', *args], capture_output=True, text=True).stdout
print(git('log', '--oneline', '--graph', '--all', '-20'))

*   0658999 Merge: develop -> main (release v2.0.0)
|\  
| *   8e94da3 Merge: docs/informe-ea3 -> develop
| |\  
| | * 3b3c3b6 docs(ea3): informe, guÃ­a de despliegue, cuaderno y README de la EA3
| * | 049e0b0 Merge: feature/deploy-render -> develop
| |\| 
| | * 4b4e8a2 ci(cd): despliegue continuo de la app web en Render (Docker + Deploy Hook)
| * | f423484 Merge: feature/webapp-fastapi -> develop
|/| | 
| |/  
| * 4a8edb4 feat(web): aplicaciÃ³n web FastAPI con API REST y tablero desplegable
|/  
*   919cc52 Merge pull request #3 from Jeremy-Pedraza/docs/informe-final
|\  
| * 9f0885b docs(informe): cifras finales, PR fusionados y captura del sitio publicado
|/  
*   5a0bfc3 Merge pull request #2 from Jeremy-Pedraza/develop
|\  
| *   9fa04fb Merge pull request #1 from Jeremy-Pedraza/docs/notebook-ea2
| |\  
| | * c11640e docs(informe): informe EA2 en PDF/DOCX, figuras y scripts de generaciÃ³n
| | * 2e97db4 ci: ejecutar integraciÃ³n continua tambiÃ©n en ramas docs/**
| | * bf49ab2 docs

In [4]:
print('Ramas:'); print(git('branch', '-a'))
print('Etiquetas:'); print(git('tag'))

Ramas:
  develop
  docs/informe-ea3
  feature/deploy-render
  feature/webapp-fastapi
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/develop
  remotes/origin/docs/informe-ea3
  remotes/origin/docs/informe-final
  remotes/origin/docs/notebook-ea2
  remotes/origin/feature/analisis-reporte
  remotes/origin/feature/cd-github-pages
  remotes/origin/feature/ci-github-actions
  remotes/origin/feature/deploy-render
  remotes/origin/feature/extraccion-producto
  remotes/origin/feature/rastreo-competencia
  remotes/origin/feature/webapp-fastapi
  remotes/origin/fix/contraste-encabezado
  remotes/origin/main

Etiquetas:


v1.0.0
v2.0.0



## Fase 2 — Pruebas automatizadas (replica la integración continua)

Las mismas 37 pruebas que ejecuta el *workflow* `ci.yml` en cada *push* y *pull request*, con cobertura. Son la puerta de calidad que debe pasar antes de desplegar.

In [5]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--cov', '--cov-report=term-missing'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print('RUFF:'); print(subprocess.run([sys.executable, '-m', 'ruff', 'check', '.'],
                                     capture_output=True, text=True).stdout or 'All checks passed!')

.....................................                                    [100%]
============================== warnings summary ===============================
..\..\venv\lib\site-packages\fastapi\testclient.py:1
  C:\Users\Desarrollador\AppData\Local\Temp\claude\f2e751d665dc1502\96d621c8-25e2-4004-af79-f64fdf40813c\scratchpad\venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=============================== tests coverage ================================
_______________ coverage: platform win32, python 3.10.6-final-0 _______________

Name                            Stmts   Miss  Cover   Missing
-------------------------------------------------------------
src\ml_scraper\__init__.py          1      0   100%
src\ml_scraper\analisis.py     

All checks passed!



## Fase 3 — La aplicación web en marcha

Se arranca la aplicación FastAPI **en proceso** con `TestClient` (sin servidor externo) y se consultan sus *endpoints*: el estado del servicio, los indicadores de posicionamiento y la ficha del producto. Es el mismo código que se despliega en la nube.

In [6]:
from fastapi.testclient import TestClient
from ml_scraper.web import app
cliente = TestClient(app)
print('GET /health ->', cliente.get('/health').json())

C:\Users\Desarrollador\AppData\Local\Temp\claude\f2e751d665dc1502\96d621c8-25e2-4004-af79-f64fdf40813c\scratchpad\venv\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


GET /health -> {'status': 'ok', 'version': '2.0.0', 'entorno': 'local', 'modo_datos': 'snapshot', 'commit': 'local'}


In [7]:
import pandas as pd
ind = cliente.get('/api/indicadores').json()
pd.DataFrame(ind.items(), columns=['Indicador', 'Valor'])

,Indicador,Valor
0,Artículos rastreados,12.00
1,Precio del artículo analizado (COP),1499900.00
2,Precio mínimo del mercado (COP),899900.00
3,Precio máximo del mercado (COP),1799900.00
4,Precio promedio del mercado (COP),1281416.00
5,Mediana del mercado (COP),1229450.00
6,Desviación vs. promedio (%),17.05
7,Ranking de precio (1 = más barato),10.00
8,Descuento promedio del mercado (%),20.90
9,Ofertas con envío gratis (%),66.70


In [8]:
comp = pd.DataFrame(cliente.get('/api/competencia').json())
print('Competidores:', len(comp))
comp[['titulo', 'precio', 'calificacion']].sort_values('precio').head(12)

Competidores: 12


,titulo,precio,calificacion
10,Tecno Camon 30 5G 256 GB Negro 8 GB RAM,899900,4.3
7,Motorola Moto G85 5G 256 GB Verde 8 GB RAM,999900,4.4
8,Honor 200 Lite 5G 256 GB Negro 8 GB RAM,1099900,4.5
11,Samsung Galaxy A55 5G 256 GB Azul Oscuro 8 GB ...,1129900,4.2
6,Motorola Edge 50 Fusion 256 GB Azul 8 GB RAM,1199000,4.5
2,Samsung Galaxy A35 5G 256 GB Azul Helado 8 GB RAM,1199900,4.6
9,Realme 13 Pro 5G 256 GB Morado 8 GB RAM,1259000,4.6
4,Xiaomi Redmi Note 14 Pro 5G 256 GB Negro 8 GB RAM,1289900,4.7
1,Samsung Galaxy A55 5G 128 GB Azul Claro 8 GB RAM,1349900,4.6
0,Samsung Galaxy A55 5G 256 GB Azul Oscuro 8 GB RAM,1499900,4.7


In [9]:
# Regresión de negocio: la conclusión de la EA1 debe mantenerse en la API desplegada.
assert abs(ind['Desviación vs. promedio (%)'] - 17.05) < 0.01
print('OK — el artículo se ofrece', ind['Desviación vs. promedio (%)'], '% por encima del promedio.')

OK — el artículo se ofrece 17.05 % por encima del promedio.


## Fase 4 — Automatización y despliegue (GitHub Actions + Render)

Los tres *workflows* del repositorio: integración continua (`ci.yml`), despliegue del reporte estático (`cd.yml`) y **despliegue del servicio en la nube** (`cd-render.yml`). Este último encadena: pruebas → build de la imagen Docker → *Deploy Hook* de Render → prueba de humo.

In [10]:
from pathlib import Path
wf = Path('.github/workflows')
for f in ['ci.yml', 'cd.yml', 'cd-render.yml']:
    print('='*70); print(f); print('='*70)
    print((wf / f).read_text(encoding='utf-8'))

ci.yml
# Integración continua: se ejecuta en cada push y en cada pull request.
# Un PR solo puede fusionarse si este workflow termina en verde (regla de protección de rama).
name: CI

on:
  push:
    branches: [main, develop, "feature/**", "fix/**", "docs/**", "release/**"]
  pull_request:
    branches: [main, develop]
  workflow_dispatch:

concurrency:
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  lint:
    name: Lint (ruff)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install ruff
      - run: ruff check . --output-format=github

  test:
    name: Pruebas (Python ${{ matrix.python-version }})
    needs: lint
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
         

In [11]:
print('Dockerfile:'); print(Path('Dockerfile').read_text(encoding='utf-8'))
print('render.yaml:'); print(Path('render.yaml').read_text(encoding='utf-8'))

Dockerfile:
# Imagen de la aplicación web (FastAPI) que se despliega como servicio en la nube (Render).
# Build:  docker build -t ml-scraper-web .
# Run:    docker run -p 8000:8000 -e APP_ENV=local ml-scraper-web
FROM python:3.12-slim

# Buenas prácticas: sin buffer en logs, sin caché de pip, sin .pyc.
ENV PYTHONUNBUFFERED=1 \
    PYTHONDONTWRITEBYTECODE=1 \
    PIP_NO_CACHE_DIR=1 \
    APP_ENV=production \
    APP_DATA_MODE=snapshot \
    APP_WORK_DIR=/tmp/ml_scraper

WORKDIR /app

# 1) Dependencias primero (capa cacheable mientras requirements.txt no cambie).
COPY requirements.txt ./
RUN pip install --no-cache-dir -r requirements.txt

# 2) Código y datos de referencia (snapshot versionado) e instalación del paquete.
COPY pyproject.toml ./
COPY src ./src
COPY data ./data
RUN pip install --no-cache-dir -e .

# 3) Usuario sin privilegios.
RUN useradd --create-home app && chown -R app:app /app
USER app

EXPOSE 8000

# Render inyecta el puerto en $PORT; en local se usa 8000 por defecto.
C

La aplicación está desplegada en **https://galaxy-a55-analytics.onrender.com**. Si el servicio está activo, la siguiente celda consulta su estado en producción (en Colab con red). En el plan gratuito el primer acceso puede tardar unos segundos por el arranque en frío.

In [12]:
import urllib.request, json
try:
    with urllib.request.urlopen('https://galaxy-a55-analytics.onrender.com/health', timeout=60) as r:
        print('Producción /health ->', json.load(r))
except Exception as e:
    print('No se pudo consultar producción (servicio dormido o sin red):', e)

No se pudo consultar producción (servicio dormido o sin red): HTTP Error 404: Not Found


## Fase 5 — Conclusiones

- El análisis dejó de ser un reporte estático y se convirtió en un **servicio web** con API REST.
- La **contenerización** garantiza que lo probado es lo desplegado; GitHub Actions es el único origen de los despliegues.
- El pipeline `test → docker → deploy → smoke` solo publica lo que superó todas las verificaciones, y la prueba de humo confirma que el servicio quedó en línea.
- La **regresión de negocio** (+17,05 %) se mantiene verificable en la API desplegada.
- Se completó el ciclo DevOps de las tres evidencias: extracción (EA1) → CI/CD y software versionado (EA2) → despliegue de un servicio en la nube (EA3).